In [12]:
import torch
from torch import nn
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader

In [13]:
BATCH_SIZE = 128
NUM_WORKERS = 0

In [14]:
def strict_size_collate(batch):
    valid_batch = [
        item for item in batch 
        if item is not None and item[0].shape == torch.Size([3, 150, 150])
    ]
    
    if len(valid_batch) == 0:
        return None
        
    return torch.utils.data.dataloader.default_collate(valid_batch)

In [15]:
train_data = ImageFolder(root='data/intel-image/train', transform=transforms.ToTensor())
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, pin_memory=False, num_workers=0, collate_fn=strict_size_collate)

test_data = ImageFolder(root='data/intel-image/test', transform=transforms.ToTensor())
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=False, num_workers=0, collate_fn=strict_size_collate)

In [16]:
train_data[0][0].shape

torch.Size([3, 150, 150])

In [17]:
train_data.class_to_idx

{'buildings': 0,
 'forest': 1,
 'glacier': 2,
 'mountain': 3,
 'sea': 4,
 'street': 5}

In [ ]:
class GenerativeVariationalAutoencoder(nn.Module):
    def __init__(self, latent_dim=1024, num_classes=len(train_data.classes)):
        super().__init__()
        self.num_classes = num_classes

        # Encoder
        self.encoder = nn.Sequential(
            # [3, 150, 150] -> [16, 75, 75]
            nn.Conv2d(in_channels=3+num_classes, out_channels=16, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True),
            # [16, 75, 75] -> [64, 38, 38]
            nn.Conv2d(in_channels=16, out_channels=64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),
            # [64, 38, 38] -> [128, 19, 19]
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(num_features=128),
            nn.ReLU(inplace=True)
        )

        self.flatten_size = 128 * 19 * 19

        # Bottleneck
        self.fc_mu = nn.Linear(self.flatten_size, latent_dim)
        self.fc_logvar = nn.Linear(self.flatten_size, latent_dim)

        # Decoder
        self.decoder_input = nn.Linear(latent_dim + num_classes, self.flatten_size)

        self.decoder = nn.Sequential(
            # [128, 19, 19] -> [64, 38, 38]
            nn.ConvTranspose2d(in_channels=128, out_channels=64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(num_features=64),
            nn.ReLU(inplace=True),
            # [64, 38, 38] -> [16, 75, 75]
            nn.ConvTranspose2d(in_channels=64, out_channels=16, kernel_size=3, stride=2, padding=1, output_padding=0),
            nn.BatchNorm2d(num_features=16),
            nn.ReLU(inplace=True),
            # [16, 75, 75] -> [3, 150, 150]
            nn.ConvTranspose2d(in_channels=16, out_channels=3, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.Sigmoid()
        )

    def encode(self, x, c):
        batch_size = x.size(0)

        c_spatial = c.view(batch_size, self.num_classes, 1, 1).expand(-1, -1, 150, 150)
        x_cond = torch.cat((x, c_spatial), dim=1)
        h = self.encoder(x_cond)
        h = h.view(batch_size, -1)

        return self.fc_mu(h), self.fc_logvar(h)

    def reparametrize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)

        return mu + eps * std

    def decode(self, z, c):
        z_cond = torch.cat([z, c], dim=1)
        h = self.decoder_input(z_cond)
        h = h.view(h.size(0), 128, 19, 19)

        return self.decoder(h)

    def forward(self, x, c):
        mu, logvar = self.encode(x, c)
        z = self.reparametrize(mu, logvar)
        reconstructed = self.decode(z, c)

        return reconstructed, mu, logvar